# 03. 이미지로 이미지 생성하기 — img2img

**img2img** 는 텍스트뿐 아니라 **입력 이미지**를 함께 주어, 그 구도와 색을 유지하며 새 이미지를 생성하는 기법입니다.
스케치를 그림으로 바꾸거나, 기존 이미지를 다른 스타일로 변환할 때 씁니다.

1. img2img 파이프라인 불러오기
2. 입력 이미지 + 프롬프트로 변환
3. `strength`(변형 정도) 비교

In [ ]:
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
print('img2img 파이프라인 준비 완료')

## 1. 입력 이미지 준비

간단한 도형을 그려 입력 이미지로 사용합니다. (실제 실습에서는 사진이나 스케치를 `Image.open(경로)`로 불러오세요.)

In [ ]:
canvas = np.full((512, 512, 3), 230, dtype=np.uint8)
canvas[180:330, 180:330] = [60, 120, 220]   # 파란 사각형
init_image = Image.fromarray(canvas)

plt.figure(figsize=(5, 5))
plt.imshow(init_image); plt.axis('off'); plt.title('입력 이미지')
plt.show()

## 2. 입력 이미지 + 프롬프트로 변환

입력 이미지의 구도를 바탕으로 프롬프트에 맞춰 변환합니다.

In [ ]:
prompt = 'a cozy wooden house in a snowy forest, digital art'
generator = torch.Generator(device=device).manual_seed(42)

result = pipe(prompt=prompt, image=init_image, strength=0.75,
              guidance_scale=7.5, generator=generator).images[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(init_image); axes[0].axis('off'); axes[0].set_title('입력')
axes[1].imshow(result); axes[1].axis('off'); axes[1].set_title('변환 결과')
plt.tight_layout(); plt.show()

## 3. strength 비교

`strength`는 입력 이미지에 노이즈를 얼마나 더할지 정합니다. 낮으면 원본에 가깝고, 높으면 프롬프트 위주로 크게 변합니다.

In [ ]:
strengths = [0.3, 0.6, 0.9]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, strengths):
    g = torch.Generator(device=device).manual_seed(42)
    img = pipe(prompt=prompt, image=init_image, strength=s,
               guidance_scale=7.5, generator=g).images[0]
    ax.imshow(img); ax.axis('off'); ax.set_title(f'strength={s}')
plt.tight_layout(); plt.show()

### 정리

- img2img로 입력 이미지의 구도를 유지하며 프롬프트에 맞춰 변환했습니다.
- `strength`가 원본 유지와 변형 정도의 균형을 조절합니다.
- 스케치→완성작, 스타일 변환, 이미지 업스케일 등 다양한 창작 작업에 활용됩니다.